# Results_paper: nighttime lights at the periodic markets

Makes two panels of Figure 5 in the paper.

| Paper exhibit | Cell | File (in `output_paper/paper_exhibits`) |
|---|---|---|
| Figure 5, panel b: markets with positive nighttime-light emissions in 2018 and 2025 | `75e7e983` | `fig05b_positive_emissions.png` |
| Figure 5, panel c: nighttime-light radiance in 2018 and 2025 (CDF) | `8ac2a286` | `fig05c_ntl_cdf.png` |

Figure 5 panel a is made by `exhibits/figures.py` (`fig05a_access_categories`).
EXHIBITS.md at the repository root lists every exhibit in the paper.

The notebook reads only from `processed_data/` (written by
`NTL_processing/06_export_processed_ntl.py`) plus the monthly VIIRS rasters, which
are needed for one thing only: the Kabul reference value in Section 3 (cached
after the first run).

**Nighttime lights convention.** All NTL here is BNR-filtered (background
noise reduction) using *each year's own* `average_masked` composite as that
year's reference mask, never a single static reference year. Light is measured
as radiance summed over valid pixels in a 10 km geodesic buffer around each market.

Cells for figures not in the final paper (other CDF variants, per-year bar and dot
plots, and an older Python event study) are in `archive/Results_paper_archive.ipynb`.

In [ ]:
import os
import sys
import glob
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate code_paper/NTL_processing/config.py whether the kernel was started in
# code_paper itself or one level above it.
_candidates = [os.getcwd(), os.path.join(os.getcwd(), "code_paper"),
               os.path.dirname(os.getcwd())]
NB_DIR = next((p for p in _candidates
               if os.path.exists(os.path.join(p, "NTL_processing", "config.py"))), None)
if NB_DIR is None:
    raise RuntimeError("Could not find NTL_processing/config.py. Start the kernel in "
                       "the code_paper folder, or set MAI_BASE.")
sys.path.insert(0, os.path.join(NB_DIR, "NTL_processing"))
sys.path.insert(0, os.path.join(NB_DIR, "exhibits"))

# Paper figures are drawn by exhibits/plots.py in the house style of exhibits/style.py;
# each figure cell below computes its data, saves it (S.save_data) and calls P.<figure>.
import style as S
import plots as P


from config import (BASE, DATA, PROCESSED, OUTPUT, PAPER_EXHIBITS, VIIRS_DIR, MASK_DIR,
                    KABUL_LON, KABUL_LAT, BUFFER_KM, MAX_RAD_CAP)

# Paper figures go to the shared paper_exhibits folder, named by exhibit number.
OUT_PAPER = PAPER_EXHIBITS

YEARS = list(range(2018, 2026))

print("base          :", BASE)
print("processed     :", PROCESSED)
print("paper exhibits:", OUT_PAPER)

## 1. Load the processed data: NTL

In [16]:
ntl = pd.read_csv(os.path.join(PROCESSED, "ntl_bnr_866markets_4years.csv"), low_memory=False)
groups = pd.read_csv(os.path.join(PROCESSED, "market_groups.csv"), low_memory=False)
panel = pd.read_stata(os.path.join(PROCESSED, "event_study_panel.dta"))

ntl = ntl.merge(groups[["mkt_uid", "group"]], on="mkt_uid", how="left")
assert ntl.group.notna().all()

print("NTL buffer stats :", ntl.shape)
print("event-study panel:", panel.shape)
print()
print(ntl.group.value_counts().to_string())

NTL buffer stats : (866, 150)
event-study panel: (1084, 32)

group
OSM only       646
Periodic       137
OSM overlap     83


## 2. Market-year light measure

For each market and year, take the **median across the twelve monthly values**
of total BNR-filtered radiance summed over the 10 km buffer. Markets with no
valid pixel in a month are missing there; a market whose median is zero or
missing is treated as dark that year.

Because this is a sum rather than a per-pixel mean, it scales with buffer
area, so anything compared against it — including the Kabul reference in the
next section — must use the same 10 km buffer.

In [17]:
def median_monthly_light(df, year):
    """Median across the 12 monthly SUM_BNR values, per market, for `year`."""
    cols = sorted(c for c in df.columns if c.startswith("SUM_BNR_%d" % year))
    assert len(cols) == 12, "%d: expected 12 monthly columns, got %d" % (year, len(cols))
    return df[cols].median(axis=1, skipna=True).fillna(0.0).clip(lower=0.0)

FIG_YEARS = [2018, 2025]
for y in FIG_YEARS :
    ntl["light_%d" % y] = median_monthly_light(ntl, y)

lit_share = pd.DataFrame({
    "n": ntl.groupby("group").size(),
    **{"positive light %d" % y: ntl.groupby("group")["light_%d" % y].apply(lambda s: (s > 0).mean())
       for y in FIG_YEARS},
})

ntl_OSM = ntl[ntl["group"].isin(['OSM overlap','OSM only'])].copy(deep=True)
ntl_OSM['group'] = 'OSM-all'

lit_share_OSM = pd.DataFrame({
    "n": ntl_OSM.groupby("group").size(),
    **{"positive light %d" % y: ntl_OSM.groupby("group")["light_%d" % y].apply(lambda s: (s > 0).mean())
       for y in FIG_YEARS},
})

lit_share = pd.concat([lit_share, lit_share_OSM])

GROUP_ORDER = ["OSM-all", "OSM only", "OSM overlap", "Periodic"]
lit_share = lit_share.reindex([g for g in GROUP_ORDER if g in lit_share.index])

pct_cols = [c for c in lit_share.columns if c != "n"]
display_tbl = pd.concat(
    [lit_share[["n"]].astype(int),
     (100 * lit_share[pct_cols]).round(1).map(lambda v: "%5.1f%%" % v)],
    axis=1,
)
display_tbl.index.name = "group"
print(display_tbl.to_string())


               n positive light 2018 positive light 2025
group                                                   
OSM-all      729               49.7%               61.6%
OSM only     646               47.7%               58.7%
OSM overlap   83               65.1%               84.3%
Periodic     137               56.9%               73.7%


## 3. Nighttime light in downtown Kabul

The CDF figures carry a reference line for Kabul. It is computed here on
exactly the same basis as the market values: a 10 km geodesic buffer around
Kabul city centre, BNR-filtered with that year's own reference mask, taking
the median across the twelve monthly buffer totals. The buffer is the same
10 km used for the markets, which is what makes a summed measure comparable.
The per-pixel mean is computed and cached alongside it for reference.

This is the one step that reads the monthly VIIRS rasters. The result is
cached to `processed_data/kabul_reference_10km_bnr.csv`, so later runs skip it.

In [18]:
from pyproj import Geod
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Polygon

GEOD = Geod(ellps="WGS84")
KABUL_CACHE = os.path.join(PROCESSED, "kabul_reference_10km_bnr.csv")


def geodesic_circle(lon, lat, radius_km=BUFFER_KM, n_points=72):
    """True geodesic circle in EPSG:4326 - same construction as the market buffers."""
    angles = np.linspace(0, 360, n_points, endpoint=False)
    lons, lats, _ = GEOD.fwd(np.full(n_points, lon), np.full(n_points, lat),
                             angles, np.full(n_points, radius_km * 1000.0))
    return Polygon(zip(lons, lats))


def kabul_monthly_stats(year):
    """Monthly BNR radiance in a 10 km buffer on Kabul city centre.

    Returns a frame indexed by YYYYMM with the buffer total ('sum') and the
    per-valid-pixel average ('mean').
    """
    geom = [geodesic_circle(KABUL_LON, KABUL_LAT)]
    with rasterio.open(os.path.join(MASK_DIR, "afg_viirs_annual_masked_%d.tif" % year)) as ds:
        mask_bool = ds.read(1) > 0

    out = {}
    tifs = sorted(f for f in glob.glob(os.path.join(VIIRS_DIR, "afg_viirs_%d*.tif" % year))
                  if "cf_cvg" not in os.path.basename(f))
    for tif in tifs:
        m = re.search(r"afg_viirs_(\d{6})\.tif$", os.path.basename(tif))
        if not m:
            continue
        with rasterio.open(tif) as ds:
            avg_rad, transform = ds.read(1), ds.transform
        if avg_rad.shape != mask_bool.shape:
            out[m.group(1)] = {"sum": np.nan, "mean": np.nan}
            continue
        valid = mask_bool & (avg_rad > 0) & (avg_rad < MAX_RAD_CAP) & ~np.isnan(avg_rad)
        bnr = np.where(valid, avg_rad, np.nan).astype("float64")
        s = zonal_stats(geom, bnr, affine=transform, stats=["sum", "mean"],
                        nodata=np.nan, all_touched=False)[0]
        out[m.group(1)] = {"sum": s["sum"], "mean": s["mean"]}
    return pd.DataFrame(out).T.astype("float64")


NEEDED = ["kabul_median_sum", "kabul_median_mean_radiance", "n_months"]

kabul = None
if os.path.exists(KABUL_CACHE):
    cached = pd.read_csv(KABUL_CACHE).set_index("year")
    if all(c in cached.columns for c in NEEDED) and set(FIG_YEARS) <= set(cached.index):
        kabul = cached
        print("loaded cached Kabul reference values")
    else:
        print("cache is stale (missing columns or years) - recomputing")

if kabul is None:
    rows = []
    for y in FIG_YEARS:
        monthly = kabul_monthly_stats(y)
        rows.append({"year": y,
                     "kabul_median_sum": float(monthly["sum"].median(skipna=True)),
                     "kabul_median_mean_radiance": float(monthly["mean"].median(skipna=True)),
                     "n_months": int(monthly["sum"].notna().sum())})
        print("%d: %d months, median buffer sum = %.1f (per-pixel mean = %.2f)"
              % (y, rows[-1]["n_months"], rows[-1]["kabul_median_sum"],
                 rows[-1]["kabul_median_mean_radiance"]))
    kabul = pd.DataFrame(rows).set_index("year")
    kabul.to_csv(KABUL_CACHE)
    print("cached to", KABUL_CACHE)

# The figures plot the buffer total, so the reference line uses the same.
KABUL_REF = kabul["kabul_median_sum"].to_dict()
print()
print("Kabul reference (buffer total, 10 km):",
      {y: round(v, 1) for y, v in KABUL_REF.items()})

loaded cached Kabul reference values

Kabul reference (buffer total, 10 km): {2018: 18464.0, 2025: 17353.2}


## 4. Figure 5, panels b and c

Both panels use the market-year light measure of Section 2 for the 137 periodic
markets, and the same definition of a lit market: median monthly light above zero.
"Unlit" in panel c is therefore the complement of the "positive emissions" share in
panel b.

**Panel c** plots the cumulative distribution on a log(1 + light) axis, so a market
with no light sits exactly at zero and every market with any positive light is to
the right of it. The curve rises at zero to the share of unlit markets and
accumulates from there across the lit markets' brightness. The vertical lines mark
the Kabul city-centre reference of Section 3, one per year.

### Access categories of the periodic markets

In [ ]:
# Access category of each periodic market, from mkt_access.csv (its `density` field),
# joined to the light measure of Section 2 (light_<year>) on orig_id == mktID.
MKT_ACCESS = os.path.join(BASE, "document", "Repo_09042026", "data", "derived", "mkt_access.csv")
DENS = {"low density": "Remote", "mid density": "Intermediate access", "high density": "High access"}
ACC_ORD = ["Remote", "Intermediate access", "High access"]

acc = pd.read_csv(MKT_ACCESS)
acc["access"] = acc.density.map(DENS)
per = ntl[ntl.group == "Periodic"].merge(acc[["mktID", "access"]],
                                         left_on="orig_id", right_on="mktID", how="left")
assert per.access.notna().all(), \
    "missing access category for %s" % per.loc[per.access.isna(), "orig_id"].tolist()
print(per.access.value_counts().reindex(ACC_ORD))

In [ ]:
# ==== PAPER FIGURE 5, panel b: markets with positive nighttime-light emissions, 2018 and 2025 ====
# Share of markets with median monthly light above zero, by access category plus all
# markets; n is in the axis labels and the number inside each bar is the count of lit markets.
# Output: paper_exhibits/fig05b_positive_emissions.png

def positive_share_table(years):
    """Lit markets (median monthly light > 0) by access category and for all markets."""
    rows = []
    for k in ACC_ORD + ["All markets"]:
        s = per if k == "All markets" else per[per.access == k]
        for y in years:
            lit = int((s["light_%d" % y] > 0).sum())
            rows.append({"group": k, "year": y, "lit": lit, "n": len(s), "pct": 100 * lit / len(s)})
    return pd.DataFrame(rows)


share_tab = positive_share_table(FIG_YEARS)
S.save_data(share_tab, "fig05b_positive_emissions")
S.save(P.fig05b_positive_emissions(share_tab), "fig05b_positive_emissions")
share_tab


In [ ]:
# ==== PAPER FIGURE 5, panel c: nighttime-light radiance in 2018 and 2025 ====
# Cumulative distribution across the 137 periodic markets, one line per year, on a
# log(1 + radiance) axis. Unlit (median monthly light == 0) is the same definition as
# panel b. Dashed vertical lines: the Kabul city-centre reference for each year.
# Output: paper_exhibits/fig05c_ntl_cdf.png
d = {"values": pd.concat([pd.DataFrame({"year": y, "mktID": per.orig_id,
                                           "light": per[f"light_{y}"].fillna(0)})
                             for y in FIG_YEARS]),
     "kabul": pd.DataFrame({"year": FIG_YEARS, "kabul_ref": [KABUL_REF[y] for y in FIG_YEARS]})}
S.save_data(d, "fig05c_ntl_cdf")
S.save(P.fig05c_ntl_cdf(d), "fig05c_ntl_cdf")
